In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
import re
import unicodedata
from scipy import stats
import geopandas as gpd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_seq_items", None)

BASE_DIR      = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data"
MENSUALES_DIR = os.path.join(BASE_DIR, "Escenarios Cambio Climatico IDEAM IV comunicacion", "Mensuales")
WEB_DATA_DIR  = os.path.join(BASE_DIR, "Scripts Python", "webpage_climate", "data")

os.chdir(BASE_DIR)
print("Directorio de trabajo :", os.getcwd())
print("Carpeta web/data      :", WEB_DATA_DIR)

In [ ]:
pip install geopandas

In [ ]:
pip install sodapy

## Precipitación histórica – Carga y procesamiento

In [ ]:
precipitacion = pd.read_csv("Precipitación_20251222.csv")

for col in ["Latitud", "Longitud", "ValorObservado"]:
    precipitacion[col] = (
        precipitacion[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace("nan", None)
        .astype(float)
    )

fecha_str_ = precipitacion["FechaObservacion"].str.slice(0, 11)
precipitacion["fecha"] = pd.to_datetime(
    fecha_str_, format="%Y %b %d", errors="coerce", cache=True
)

In [ ]:
station_cols = [
    "CodigoEstacion", "NombreEstacion", "Departamento",
    "Municipio", "ZonaHidrografica", "Latitud", "Longitud"
]

df_daily = (
    precipitacion
    .groupby(station_cols + ["fecha"], as_index=False)
    .agg(
        precip_min_10min=("ValorObservado", "min"),
        precip_max_10min=("ValorObservado", "max"),
        precip_media_10min=("ValorObservado", "mean"),
        precip_acum_diaria=("ValorObservado", "sum")
    )
)

df_daily = df_daily.sort_values(["CodigoEstacion", "fecha"])
print(df_daily.shape)
df_daily.head()

In [ ]:
data_2 = pd.read_csv("Precipitación_20251222_2.csv", decimal=",")

for col in ["Latitud", "Longitud", "ValorObservado"]:
    data_2[col] = data_2[col].astype(float)

fecha_str = data_2["FechaObservacion"].str.slice(0, 11)
data_2["fecha"] = pd.to_datetime(
    fecha_str, format="%Y %b %d", errors="coerce", cache=True
)

In [ ]:
df_daily_2 = (
    data_2
    .groupby(station_cols + ["fecha"], as_index=False)
    .agg(
        precip_min_10min=("ValorObservado", "min"),
        precip_max_10min=("ValorObservado", "max"),
        precip_media_10min=("ValorObservado", "mean"),
        precip_acum_diaria=("ValorObservado", "sum")
    )
)

df_daily_2 = df_daily_2.sort_values(["CodigoEstacion", "fecha"])
print(df_daily_2.shape)
df_daily_2.head()

In [20]:
df_daily_vf = pd.concat([df_daily, df_daily_2])
df_daily_vf = df_daily_vf.drop_duplicates(["CodigoEstacion", "fecha"])
print("df_daily_vf shape:", df_daily_vf.shape)
df_daily_vf.head()

df_daily_vf shape: (976941, 12)


,CodigoEstacion,NombreEstacion,Departamento,Municipio,ZonaHidrografica,Latitud,Longitud,fecha,precip_min_10min,precip_max_10min,precip_media_10min,precip_acum_diaria
0,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-19,0.0,0.1,0.000526,0.1
1,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-20,0.0,0.0,0.000000,0.0
2,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-21,0.0,0.1,0.000347,0.1
3,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-22,0.0,0.1,0.000347,0.1
4,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.888719,-76.145167,2022-12-23,0.0,0.0,0.000000,0.0


In [21]:
df_daily_vf['fecha'] = pd.to_datetime(df_daily_vf['fecha'])
df_daily_vf['mes']   = df_daily_vf['fecha'].dt.month

# ── 1. Percentil 95 por estación y mes (umbral histórico) ─────────────────
p95 = (
    df_daily_vf
    .groupby(['CodigoEstacion', 'mes'])['precip_acum_diaria']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'precip_acum_diaria': 'p95'})
)

df_daily_vf = df_daily_vf.merge(p95, on=['CodigoEstacion', 'mes'], how='left')
df_daily_vf['extremo'] = (df_daily_vf['precip_acum_diaria'] > df_daily_vf['p95']).astype(int)

# ── 2. Frecuencia histórica de extremos por estación ─────────────────────
station_cols_alerta = ['CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio', 'Latitud', 'Longitud']

estaciones_alerta = (
    df_daily_vf
    .groupby(station_cols_alerta)['extremo']
    .mean()
    .reset_index()
    .rename(columns={'extremo': 'frecuencia_extremos'})
)

# ── 3. Tendencia en los últimos 2 años ────────────────────────────────────
fecha_max   = df_daily_vf['fecha'].max()
fecha_corte = fecha_max - pd.DateOffset(years=2)
df_reciente = df_daily_vf[df_daily_vf['fecha'] >= fecha_corte].copy()

df_reciente['anio_mes'] = df_reciente['fecha'].dt.to_period('M').dt.to_timestamp()

extremos_mensuales = (
    df_reciente
    .groupby(['CodigoEstacion', 'anio_mes'])['extremo']
    .agg(n_extremos='sum', n_dias='count')
    .reset_index()
)
extremos_mensuales['tasa_extremos'] = extremos_mensuales['n_extremos'] / extremos_mensuales['n_dias']

def calcular_tendencia(grupo):
    """Regresión lineal de la tasa mensual de extremos en el tiempo."""
    if len(grupo) < 4:
        return pd.Series({'pendiente': 0.0, 'p_valor': 1.0, 'tendencia': 'insuficiente'})
    x = np.arange(len(grupo))
    y = grupo['tasa_extremos'].values
    slope, _, _, p_value, _ = stats.linregress(x, y)
    if p_value < 0.05:
        direccion = 'creciente' if slope > 0 else 'decreciente'
    else:
        direccion = 'estable'
    return pd.Series({'pendiente': slope, 'p_valor': p_value, 'tendencia': direccion})

tendencias = (
    extremos_mensuales
    .groupby('CodigoEstacion')
    .apply(calcular_tendencia)
    .reset_index()
)

# ── 4. Ratio reciente vs histórico ────────────────────────────────────────
freq_reciente = (
    extremos_mensuales
    .groupby('CodigoEstacion')['tasa_extremos']
    .mean()
    .reset_index()
    .rename(columns={'tasa_extremos': 'frecuencia_reciente'})
)

estaciones_alerta = (
    estaciones_alerta
    .merge(tendencias[['CodigoEstacion', 'pendiente', 'p_valor', 'tendencia']], on='CodigoEstacion', how='left')
    .merge(freq_reciente, on='CodigoEstacion', how='left')
)

estaciones_alerta['ratio_reciente'] = (
    estaciones_alerta['frecuencia_reciente']
    / estaciones_alerta['frecuencia_extremos'].replace(0, np.nan)
)

# ── 5. Alerta compuesta (4 niveles) ───────────────────────────────────────
UMBRAL_FREQ   = 0.10   # >10 % de días históricos sobre p95
UMBRAL_RATIO  = 1.20   # últimos 2 años tienen ≥20 % más extremos que el histórico

def categorizar_alerta(row):
    alta_freq       = row['frecuencia_extremos'] > UMBRAL_FREQ
    creciente       = row['tendencia'] == 'creciente'
    aceleracion     = pd.notna(row['ratio_reciente']) and row['ratio_reciente'] > UMBRAL_RATIO
    if alta_freq and (creciente or aceleracion):
        return 'CRÍTICA'
    elif alta_freq:
        return 'ALTA'
    elif creciente or aceleracion:
        return 'MODERADA'
    else:
        return 'BAJA'

NIVEL_NUMERICO = {'BAJA': 0, 'MODERADA': 1, 'ALTA': 2, 'CRÍTICA': 3}

estaciones_alerta['alerta_compuesta'] = estaciones_alerta.apply(categorizar_alerta, axis=1)
estaciones_alerta['alerta']           = estaciones_alerta['alerta_compuesta'].map(NIVEL_NUMERICO)

# ── 6. Resumen ────────────────────────────────────────────────────────────
print("=== Distribución de alerta compuesta ===")
print(estaciones_alerta['alerta_compuesta'].value_counts())
print("\n=== Tendencias últimos 2 años ===")
print(estaciones_alerta['tendencia'].value_counts())

cols_resumen = [
    'CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio',
    'frecuencia_extremos', 'frecuencia_reciente', 'ratio_reciente',
    'tendencia', 'p_valor', 'alerta_compuesta', 'alerta'
]
estaciones_alerta[cols_resumen].sort_values('alerta', ascending=False).head(5)

=== Distribución de alerta compuesta ===
alerta_compuesta
BAJA        2017
MODERADA     700
ALTA         276
CRÍTICA       60
Name: count, dtype: int64

=== Tendencias últimos 2 años ===
tendencia
insuficiente    1101
estable         1016
creciente        192
decreciente      113
Name: count, dtype: int64


,CodigoEstacion,NombreEstacion,Departamento,Municipio,frecuencia_extremos,frecuencia_reciente,ratio_reciente,tendencia,p_valor,alerta_compuesta,alerta
406,21206780,LA INDEPENDENCI,BOGOTÁ,BOGOTÁ D.C,0.138743,0.099329,0.715921,creciente,0.029645,CRÍTICA,3
411,21206810,SAN BENITO,BOGOTÁ,BOGOTÁ D.C,0.144414,0.097141,0.672655,creciente,0.000185,CRÍTICA,3
414,21206890,CERRO CAZADORES,BOGOTÁ,BOGOTÁ D.C,0.162304,0.105001,0.646941,creciente,0.012505,CRÍTICA,3
876,26027200,PUENTE CARRETERA,CAUCA,POPAYÁN,0.107143,0.166667,1.555556,insuficiente,1.000000,CRÍTICA,3
879,26055100,FARALLONES,VALLE DEL CAUCA,CALI,0.131429,0.114708,0.872779,creciente,0.000819,CRÍTICA,3


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INDICADOR DE SEQUÍA – Días Consecutivos Secos (CDD)
# ══════════════════════════════════════════════════════════════════════════════

UMBRAL_DIA_SECO = 1.0  # < 1 mm/día = día seco (estándar OMM/IDEAM)

# ── 1. Extraer rachas de días consecutivos secos por estación ─────────────
def extraer_rachas_secas(grupo):
    """Devuelve DataFrame (CodigoEstacion, fecha_inicio, duracion_dias) por racha seca."""
    cod   = grupo['CodigoEstacion'].iloc[0]
    grupo = grupo.sort_values('fecha').reset_index(drop=True)
    rachas, racha, inicio = [], 0, None
    for _, row in grupo.iterrows():
        if row['precip_acum_diaria'] < UMBRAL_DIA_SECO:
            if racha == 0:
                inicio = row['fecha']
            racha += 1
        else:
            if racha > 0:
                rachas.append({'fecha_inicio': inicio, 'duracion_dias': racha})
            racha = 0
    if racha > 0:
        rachas.append({'fecha_inicio': inicio, 'duracion_dias': racha})
    if rachas:
        df = pd.DataFrame(rachas)
    else:
        df = pd.DataFrame({'fecha_inicio': pd.Series(dtype='datetime64[ns]'),
                           'duracion_dias': pd.Series(dtype='int64')})
    df['CodigoEstacion'] = cod
    return df

rachas_por_estacion = (
    df_daily_vf
    .groupby('CodigoEstacion', group_keys=False)
    .apply(extraer_rachas_secas)
    .reset_index(drop=True)
)
# Garantiza dtype numérico tras el concat de todos los grupos
rachas_por_estacion['duracion_dias'] = pd.to_numeric(rachas_por_estacion['duracion_dias'])

# ── 2. Percentil 90 histórico de duración de rachas por estación ──────────
p90_rachas = (
    rachas_por_estacion
    .groupby('CodigoEstacion')['duracion_dias']
    .quantile(0.90)
    .reset_index()
    .rename(columns={'duracion_dias': 'p90_racha'})
)

rachas_por_estacion = rachas_por_estacion.merge(p90_rachas, on='CodigoEstacion', how='left')
rachas_por_estacion['sequia_extrema'] = (
    rachas_por_estacion['duracion_dias'] > rachas_por_estacion['p90_racha']
).astype(int)

# ── 3. Frecuencia histórica de rachas extremas por estación ───────────────
freq_sequia_hist = (
    rachas_por_estacion
    .groupby('CodigoEstacion')['sequia_extrema']
    .mean()
    .reset_index()
    .rename(columns={'sequia_extrema': 'freq_sequia_hist'})
)

# ── 4. Frecuencia reciente (últimos 2 años) de rachas extremas ────────────
fecha_max_s   = df_daily_vf['fecha'].max()
fecha_corte_s = fecha_max_s - pd.DateOffset(years=2)

rachas_rec = rachas_por_estacion[
    rachas_por_estacion['fecha_inicio'] >= fecha_corte_s
].copy()

freq_sequia_rec = (
    rachas_rec
    .groupby('CodigoEstacion')['sequia_extrema']
    .mean()
    .reset_index()
    .rename(columns={'sequia_extrema': 'freq_sequia_rec'})
)

# ── 5. Tendencia mensual (regresión lineal) ───────────────────────────────
rachas_rec['anio_mes'] = rachas_rec['fecha_inicio'].dt.to_period('M').dt.to_timestamp()

def tasa_mensual_sequia(df_racha):
    """Fracción mensual de rachas extremas; incluye CodigoEstacion en la salida."""
    cod = df_racha['CodigoEstacion'].iloc[0]
    result = (
        df_racha
        .groupby('anio_mes')['sequia_extrema']
        .mean()
        .reset_index()
        .rename(columns={'sequia_extrema': 'tasa_sequia'})
    )
    result['CodigoEstacion'] = cod
    return result

tasa_mensual_por_est = (
    rachas_rec
    .groupby('CodigoEstacion', group_keys=False)
    .apply(tasa_mensual_sequia)
    .reset_index(drop=True)
)

def calcular_tendencia_sequia(grupo):
    if len(grupo) < 4:
        return pd.Series({'pendiente_seq': 0.0, 'tendencia_sequia': 'insuficiente'})
    x = np.arange(len(grupo))
    y = grupo['tasa_sequia'].values
    slope, _, _, p_value, _ = stats.linregress(x, y)
    direccion = 'estable'
    if p_value < 0.05:
        direccion = 'creciente' if slope > 0 else 'decreciente'
    return pd.Series({'pendiente_seq': slope, 'tendencia_sequia': direccion})

tendencias_sequia = (
    tasa_mensual_por_est
    .groupby('CodigoEstacion')
    .apply(calcular_tendencia_sequia)
    .reset_index()
)

# ── 6. Ratio reciente vs histórico de sequías ─────────────────────────────
ratio_sequia = freq_sequia_hist.merge(freq_sequia_rec, on='CodigoEstacion', how='left')
ratio_sequia['ratio_sequia'] = (
    ratio_sequia['freq_sequia_rec']
    / ratio_sequia['freq_sequia_hist'].replace(0, np.nan)
)

# ── 7. Categorizar sequía ─────────────────────────────────────────────────
UMBRAL_FREQ_SEQ  = 0.10   # >10% de rachas históricas son extremas
UMBRAL_RATIO_SEQ = 1.20   # últimos 2 años ≥20% más rachas extremas que el histórico

estaciones_sequia = (
    freq_sequia_hist
    .merge(ratio_sequia[['CodigoEstacion', 'freq_sequia_rec', 'ratio_sequia']], on='CodigoEstacion', how='left')
    .merge(tendencias_sequia[['CodigoEstacion', 'tendencia_sequia']], on='CodigoEstacion', how='left')
)

def categorizar_sequia(row):
    alta_freq   = row['freq_sequia_hist'] > UMBRAL_FREQ_SEQ
    creciente   = row['tendencia_sequia'] == 'creciente'
    aceleracion = pd.notna(row['ratio_sequia']) and row['ratio_sequia'] > UMBRAL_RATIO_SEQ
    if alta_freq and (creciente or aceleracion):
        return 'SEVERA'
    elif alta_freq:
        return 'MODERADA'
    elif creciente or aceleracion:
        return 'LEVE'
    else:
        return 'NORMAL'

NIVEL_SEQUIA = {'NORMAL': 0, 'LEVE': 1, 'MODERADA': 2, 'SEVERA': 3}

estaciones_sequia['sequia_categoria'] = estaciones_sequia.apply(categorizar_sequia, axis=1)
estaciones_sequia['sequia_nivel']     = estaciones_sequia['sequia_categoria'].map(NIVEL_SEQUIA)

# ── 8. Incorporar sequía en estaciones_alerta ─────────────────────────────
estaciones_alerta = estaciones_alerta.merge(
    estaciones_sequia[['CodigoEstacion', 'freq_sequia_hist', 'freq_sequia_rec',
                        'ratio_sequia', 'tendencia_sequia', 'sequia_categoria', 'sequia_nivel']],
    on='CodigoEstacion', how='left'
)
estaciones_alerta['sequia_categoria'] = estaciones_alerta['sequia_categoria'].fillna('NORMAL')
estaciones_alerta['sequia_nivel']     = estaciones_alerta['sequia_nivel'].fillna(0).astype(int)

# ── 9. Tipo de alerta dominante por estación ──────────────────────────────
# Umbrales estrictos: solo eventos con evidencia histórica sólida
# lluvia: ALTA(2) o CRÍTICA(3) — requiere alta freq histórica
# sequía: MODERADA(2) o SEVERA(3) — requiere alta freq histórica de rachas largas
def tipo_alerta_estacion(row):
    lluvia = row['alerta'] >= 2        # solo ALTA o CRÍTICA
    sequia = row['sequia_nivel'] >= 2  # solo MODERADA o SEVERA
    if lluvia and sequia:
        return 'Ambos'
    elif lluvia:
        return 'Exceso de lluvias'
    elif sequia:
        return 'Sequía recurrente'
    else:
        return 'Sin alerta'

estaciones_alerta['tipo_alerta'] = estaciones_alerta.apply(tipo_alerta_estacion, axis=1)

# ── 10. Resumen ───────────────────────────────────────────────────────────
print("=== Alerta lluvia (estaciones) ===")
print(estaciones_alerta['alerta_compuesta'].value_counts())
print("\n=== Categoría sequía (estaciones) ===")
print(estaciones_alerta['sequia_categoria'].value_counts())
print("\n=== Tipo de alerta dominante (estaciones) ===")
print(estaciones_alerta['tipo_alerta'].value_counts())

In [ ]:
# Carga davipola y proyecta a EPSG 3116
davipola = pd.read_excel(os.path.join(MENSUALES_DIR, "davipola_dane.xlsx"))

gdf_mun = gpd.GeoDataFrame(
    davipola,
    geometry=gpd.points_from_xy(davipola.LONGITUD, davipola.LATITUD),
    crs="EPSG:4326",
).to_crs(epsg=3116)

# Convierte estaciones_alerta a GeoDataFrame y proyecta
gdf_estaciones = gpd.GeoDataFrame(
    estaciones_alerta,
    geometry=gpd.points_from_xy(estaciones_alerta.Longitud, estaciones_alerta.Latitud),
    crs="EPSG:4326"
).to_crs(epsg=3116)

# Asigna el municipio más cercano a cada estación
gdf_est_muni = gpd.sjoin_nearest(
    gdf_mun[['COD_DPTO', 'NOM_DPTO', 'COD_MPIO', 'NOM_MPIO', 'geometry']],
    gdf_estaciones,
    how='right',
    distance_col="dist_m"
)

# ── Helpers robustos ante NaN ──────────────────────────────────────────────
def modo_seguro(serie, default='BAJA'):
    """Moda ignorando NaN; si todo es NaN retorna default."""
    vc = serie.dropna().value_counts()
    return vc.idxmax() if not vc.empty else default

def tendencia_muni(serie):
    """Prioriza 'creciente'; si no, toma la moda; si vacío, 'sin_datos'."""
    vals = serie.dropna()
    if vals.empty:
        return 'sin_datos'
    if 'creciente' in vals.values:
        return 'creciente'
    vc = vals.value_counts()
    return vc.idxmax() if not vc.empty else 'sin_datos'

def modo_sequia(serie):
    return modo_seguro(serie, default='NORMAL')

def tipo_alerta_muni(serie):
    """Determina tipo de alerta dominante a nivel municipal."""
    vals = serie.dropna()
    if vals.empty:
        return 'Sin alerta'
    # Prioriza "Ambos" si alguna estación lo tiene
    if 'Ambos' in vals.values:
        return 'Ambos'
    # Luego el más frecuente
    vc = vals.value_counts()
    return vc.idxmax()

# ── Agrega alertas a nivel municipal ──────────────────────────────────────
df_alerta_municipal = (
    gdf_est_muni
    .groupby(['COD_DPTO', 'NOM_DPTO', 'COD_MPIO', 'NOM_MPIO'], as_index=False)
    .agg(
        alerta_compuesta=('alerta_compuesta', modo_seguro),
        frecuencia_extremos=('frecuencia_extremos', 'max'),
        frecuencia_reciente=('frecuencia_reciente', 'max'),
        ratio_reciente=('ratio_reciente', 'max'),
        tendencia=('tendencia', tendencia_muni),
        sequia_categoria=('sequia_categoria', modo_sequia),
        freq_sequia_hist=('freq_sequia_hist', 'max'),
        freq_sequia_rec=('freq_sequia_rec', 'max'),
        tendencia_sequia=('tendencia_sequia', tendencia_muni),
        tipo_alerta=('tipo_alerta', tipo_alerta_muni),
        n_estaciones=('CodigoEstacion', 'count'),
    )
)

# Derivar alerta numérica desde alerta_compuesta (consistencia garantizada)
NIVEL_NUMERICO = {'BAJA': 0, 'MODERADA': 1, 'ALTA': 2, 'CRÍTICA': 3}
NIVEL_SEQUIA   = {'NORMAL': 0, 'LEVE': 1, 'MODERADA': 2, 'SEVERA': 3}

df_alerta_municipal['alerta']       = df_alerta_municipal['alerta_compuesta'].map(NIVEL_NUMERICO)
df_alerta_municipal['sequia_nivel'] = df_alerta_municipal['sequia_categoria'].map(NIVEL_SEQUIA)

print(f"Municipios con cobertura de estaciones: {len(df_alerta_municipal):,}")
print("\n=== Alerta lluvia municipal (alerta_compuesta) ===")
print(df_alerta_municipal['alerta_compuesta'].value_counts())
print("\n=== Categoría sequía municipal (sequia_categoria) ===")
print(df_alerta_municipal['sequia_categoria'].value_counts())
print("\n=== Tipo de alerta dominante municipal ===")
print(df_alerta_municipal['tipo_alerta'].value_counts())

# ── Guardar salidas ───────────────────────────────────────────────────────
# 1. Nivel estación → consumido por datos precipitacion diarios.ipynb
estaciones_alerta['cod_norm'] = estaciones_alerta['CodigoEstacion'].astype(str).str.strip().str.lstrip('0')
out_estaciones = os.path.join(WEB_DATA_DIR, "alerta_historica_estaciones.csv")
estaciones_alerta.to_csv(out_estaciones, index=False, encoding="utf-8-sig")
print("\nGuardado (estaciones):", out_estaciones)

# 2. Nivel municipal → salida principal de este notebook
out_municipal = os.path.join(WEB_DATA_DIR, "alerta_historica_municipal.csv")
df_alerta_municipal.to_csv(out_municipal, index=False, encoding="utf-8-sig")
print("Guardado (municipal) :", out_municipal)

df_alerta_municipal.sort_values('alerta', ascending=False).head(8)

In [24]:
df_alerta_municipal["alerta"].value_counts()

alerta
0    216
1    175
2    148
3     48
Name: count, dtype: int64